# Day 18 — Method Comparison and Candidate Shortlisting

## Objective

The objective of Day 18 is to compare the anomaly-detection and forecasting methods developed so far, analyse their behaviour and limitations, and shortlist candidate methods for deeper experiments.

## Methods Developed

### Anomaly Detection
- Statistical Z-score baseline
- Isolation Forest baseline
- Tuned Isolation Forest candidate

### Forecasting
- Persistence baseline
- Random Forest Regressor

## Evaluation

For anomaly detection, anomaly rate and validation behaviour are used because ground-truth anomaly labels are currently unavailable.

For forecasting, MAE and RMSE are used on the chronological test holdout.

Processing time will be recorded where available.

## Analysis Areas

The comparison will consider:

- Performance
- Validation behaviour
- Overfitting risk
- Dataset limitations
- Model complexity
- Failure patterns

## Candidate Selection

The strongest candidate methods will be shortlisted for deeper experiments based on their observed validation behaviour, performance, stability, and suitability for the available dataset.

In [1]:
import pandas as pd
import numpy as np

# Day 18 — Method comparison

comparison = pd.DataFrame({
    "Method": [
        "Z-score Statistical Baseline",
        "Isolation Forest Baseline",
        "Tuned Isolation Forest",
        "Persistence Forecast",
        "Random Forest Forecast"
    ],
    "Task": [
        "Anomaly Detection",
        "Anomaly Detection",
        "Anomaly Detection",
        "Forecasting",
        "Forecasting"
    ],
    "Evaluation Metric": [
        "Anomaly Rate",
        "Anomaly Rate",
        "Validation Anomaly Rate",
        "MAE / RMSE",
        "MAE / RMSE"
    ],
    "Status": [
        "Baseline",
        "Baseline",
        "Candidate",
        "Baseline",
        "Candidate"
    ]
})

comparison

,Method,Task,Evaluation Metric,Status
0,Z-score Statistical Baseline,Anomaly Detection,Anomaly Rate,Baseline
1,Isolation Forest Baseline,Anomaly Detection,Anomaly Rate,Baseline
2,Tuned Isolation Forest,Anomaly Detection,Validation Anomaly Rate,Candidate
3,Persistence Forecast,Forecasting,MAE / RMSE,Baseline
4,Random Forest Forecast,Forecasting,MAE / RMSE,Candidate


In [4]:
# Day 18 — Forecasting results from Day 16

forecast_results = pd.DataFrame({
    "Method": [
        "Persistence Forecast",
        "Random Forest Forecast"
    ],
    "Test MAE": [
        1.133537,
        0.981553
    ],
    "Test RMSE": [
        1.609520,
        1.245266
    ]
})

forecast_results

,Method,Test MAE,Test RMSE
0,Persistence Forecast,1.133537,1.609520
1,Random Forest Forecast,0.981553,1.245266


In [6]:
# Day 17 — Recreate candidate anomaly results

anomaly_results = pd.DataFrame({
    "Model": [
        "IF_3pct",
        "IF_5pct",
        "IF_10pct"
    ],
    "Train Anomaly Rate": [
        0.03,
        0.05,
        0.10
    ],
    "Validation Anomaly Rate": [
        0.03,
        0.05,
        0.10
    ]
})

anomaly_results["Rate Difference"] = abs(
    anomaly_results["Train Anomaly Rate"] -
    anomaly_results["Validation Anomaly Rate"]
)

anomaly_results["Train Anomaly Rate"] = (
    anomaly_results["Train Anomaly Rate"] * 100
).round(2)

anomaly_results["Validation Anomaly Rate"] = (
    anomaly_results["Validation Anomaly Rate"] * 100
).round(2)

anomaly_results["Rate Difference"] = (
    anomaly_results["Rate Difference"] * 100
).round(2)

anomaly_results

,Model,Train Anomaly Rate,Validation Anomaly Rate,Rate Difference
0,IF_3pct,3.0,3.0,0.0
1,IF_5pct,5.0,5.0,0.0
2,IF_10pct,10.0,10.0,0.0


In [7]:
import joblib
import os

model_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\models\day17_isolation_forest_candidate.joblib"

saved_candidate = joblib.load(model_path)

candidate_model = saved_candidate["model"]
imputer = saved_candidate["imputer"]
sensor_cols = saved_candidate["features"]
candidate_name = saved_candidate["model_name"]

print("Candidate model:", candidate_name)
print("Features:", sensor_cols)

Candidate model: IF_3pct
Features: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']


In [9]:
# Day 18 — Load datasets and Day 17 candidate model

import pandas as pd
import numpy as np
import joblib

base_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed"

# Load engineered datasets
train_df = pd.read_csv(base_path + r"\train_v1_engineered.csv")
val_df = pd.read_csv(base_path + r"\validation_v1_engineered.csv")
test_df = pd.read_csv(base_path + r"\test_v1_engineered.csv")

# Convert timestamps
for data in [train_df, val_df, test_df]:
    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        errors="coerce"
    )
    data.sort_values("timestamp", inplace=True)
    data.reset_index(drop=True, inplace=True)

# Load Day 17 candidate model
model_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\models\day17_isolation_forest_candidate.joblib"

saved_candidate = joblib.load(model_path)

candidate_model = saved_candidate["model"]
imputer = saved_candidate["imputer"]
sensor_cols = saved_candidate["features"]
candidate_name = saved_candidate["model_name"]

print("Candidate model:", candidate_name)
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Sensor features:", len(sensor_cols))

Candidate model: IF_3pct
Train: (2291, 42)
Validation: (491, 42)
Test: (491, 42)
Sensor features: 9


In [10]:
# Day 17 candidate — actual anomaly rates

X_train = train_df[sensor_cols]
X_val = val_df[sensor_cols]
X_test = test_df[sensor_cols]

X_train_imp = imputer.transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

train_predictions = candidate_model.predict(X_train_imp)
val_predictions = candidate_model.predict(X_val_imp)
test_predictions = candidate_model.predict(X_test_imp)

train_rate = np.mean(train_predictions == -1) * 100
val_rate = np.mean(val_predictions == -1) * 100
test_rate = np.mean(test_predictions == -1) * 100

actual_anomaly_results = pd.DataFrame({
    "Model": [candidate_name],
    "Train Anomaly Rate (%)": [round(train_rate, 2)],
    "Validation Anomaly Rate (%)": [round(val_rate, 2)],
    "Test Anomaly Rate (%)": [round(test_rate, 2)]
})

actual_anomaly_results

,Model,Train Anomaly Rate (%),Validation Anomaly Rate (%),Test Anomaly Rate (%)
0,IF_3pct,3.01,3.05,1.43


In [11]:
# Day 18 — Final Method Comparison

final_comparison = pd.DataFrame({
    "Method": [
        "Z-score Statistical Baseline",
        "Isolation Forest Baseline",
        "Tuned Isolation Forest",
        "Persistence Forecast",
        "Random Forest Forecast"
    ],
    "Task": [
        "Anomaly Detection",
        "Anomaly Detection",
        "Anomaly Detection",
        "Forecasting",
        "Forecasting"
    ],
    "Performance": [
        "Anomaly rate; no labels",
        "5% contamination baseline",
        f"Validation anomaly rate: {val_rate:.2f}%",
        "MAE 1.1335 | RMSE 1.6095",
        "MAE 0.9816 | RMSE 1.2453"
    ],
    "Complexity": [
        "Low",
        "Medium",
        "Medium",
        "Very Low",
        "Medium"
    ],
    "Status": [
        "Baseline",
        "Baseline",
        "Candidate",
        "Baseline",
        "Candidate"
    ]
})

final_comparison

,Method,Task,Performance,Complexity,Status
0,Z-score Statistical Baseline,Anomaly Detection,Anomaly rate; no labels,Low,Baseline
1,Isolation Forest Baseline,Anomaly Detection,5% contamination baseline,Medium,Baseline
2,Tuned Isolation Forest,Anomaly Detection,Validation anomaly rate: 3.05%,Medium,Candidate
3,Persistence Forecast,Forecasting,MAE 1.1335 | RMSE 1.6095,Very Low,Baseline
4,Random Forest Forecast,Forecasting,MAE 0.9816 | RMSE 1.2453,Medium,Candidate


# Day 18 — 3-Slide Progress Pack

## Slide 1 — Methods Developed

### Anomaly Detection
- Statistical Z-score baseline
- Isolation Forest baseline
- Tuned Isolation Forest candidate

### Forecasting
- Persistence baseline
- Random Forest regression

### Dataset
- 3 chronological splits
- 2,291 training observations
- 491 validation observations
- 491 test observations
- 9 available water-quality sensor variables

---

## Slide 2 — Method Comparison

### Anomaly Detection
- Ground-truth anomaly labels are unavailable.
- Evaluation therefore uses anomaly rates, validation behaviour and score distributions.
- Isolation Forest provides a flexible non-linear anomaly-detection approach.
- The tuned Isolation Forest candidate was evaluated on training, validation and test data.

### Forecasting

| Model | Test MAE | Test RMSE |
|---|---:|---:|
| Persistence | 1.1335 | 1.6095 |
| Random Forest | 0.9816 | 1.2453 |

Random Forest achieved lower MAE and RMSE than the persistence baseline on the chronological test holdout.

---

## Slide 3 — Limitations and Shortlisted Candidates

### Key Limitations
- Dataset is relatively small.
- No ground-truth anomaly labels are available.
- Timestamps were synthetically generated.
- Temperature, dissolved oxygen and pressure are unavailable.
- Forecasting currently focuses on pH.

### Failure Patterns
- Anomaly models may flag legitimate extreme measurements.
- Changes in feature distribution may affect anomaly rates.
- Forecasting models may struggle with sudden changes or extreme pH values.

### Shortlisted Candidates

**1. Tuned Isolation Forest**
- Candidate for deeper anomaly-detection experiments.
- Validation behaviour will be further investigated.

**2. Random Forest Regressor**
- Candidate for deeper forecasting experiments.
- Current test performance: MAE 0.9816 and RMSE 1.2453.

### Next Phase
Perform robustness experiments and investigate whether additional real temporal data or validated anomaly labels can improve the system.